In [1]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))


In [2]:
import numpy as np
from qiskit.quantum_info import Statevector
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score
from src.data_load import load_data
from src.preprocess import preprocess

In [4]:
features = 4

X_train, X_test, y_train, y_test = load_data()
_, _, X_train_pca, X_test_pca, pca = preprocess(X_train, X_test, features)

In [5]:
def normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def report(name, pred):
    print(f"{name}: acc={accuracy_score(y_test, pred):.3f}  "
          f"recall(malignant)={recall_score(y_test, pred, pos_label=0):.3f}  "
          f"F1={f1_score(y_test, pred, average='macro'):.3f}")

In [6]:
mm = MinMaxScaler(feature_range=(0.1, 1))
X_train_amp = normalize(mm.fit_transform(X_train_pca))
X_test_amp = normalize(np.clip(mm.transform(X_test_pca), 0.1, 1))

In [7]:
S_train = np.array([Statevector(r).data for r in X_train_amp])
S_test = np.array([Statevector(r).data for r in X_test_amp])
K_train = np.abs(S_train.conj() @ S_train.T) ** 2
K_test = np.abs(S_test.conj() @ S_train.T) ** 2

In [8]:
qsvm = SVC(kernel="precomputed", C=1.0).fit(K_train, y_train)
report("Amplitude QSVM (C=1)", qsvm.predict(K_test))

Amplitude QSVM (C=1): acc=0.956  recall(malignant)=0.929  F1=0.953


In [9]:
# Control 1: same kernel written classically
poly = SVC(kernel="poly", degree=2, gamma=1, coef0=0, C=1.0).fit(X_train_amp, y_train)
report("Poly deg-2 control (C=1)", poly.predict(X_test_amp))

Poly deg-2 control (C=1): acc=0.956  recall(malignant)=0.929  F1=0.953


In [10]:
# check on the kernel formula itself
print("Kernel formula matches:",
      np.isclose(K_train[0, 1], np.dot(X_train_amp[0], X_train_amp[1]) ** 2))

Kernel formula matches: True


In [11]:
# Control 2: linear SVM on the standard PCA-4 input (no shift, no normalization)
lin = SVC(kernel="linear", C=1.0).fit(X_train_pca, y_train)
report("Linear SVM, PCA 4 (C=1)", lin.predict(X_test_pca))

Linear SVM, PCA 4 (C=1): acc=0.965  recall(malignant)=0.929  F1=0.962


In [12]:
# Control 3: poly control on the CENTERED normalized input (the 63% failure case)
X_train_c, X_test_c = normalize(X_train_pca), normalize(X_test_pca)
poly_c = SVC(kernel="poly", degree=2, gamma=1, coef0=0, C=1.0).fit(X_train_c, y_train)
report("Poly deg-2, centered input (C=1)", poly_c.predict(X_test_c))

Poly deg-2, centered input (C=1): acc=0.623  recall(malignant)=0.286  F1=0.546
